# BERT Fine-Tuning for Sentiment Analysis

This notebook fine-tunes both bert-base-cased and bert-base-uncased on the same Cornell Movie Review Dataset used in the classical pipeline notebook, enabling direct comparison between bag-of-words approaches and contextual transformer embeddings.

**Key findings:** BERT Uncased achieves 0.91 accuracy and F1 on the held-out test set, a 6-point gain over the best classical model (SVM, 0.85 F1). Training loss decreases ~72% over 1,750 steps, indicating stable convergence. Uncased outperforms Cased (0.91 vs 0.89), consistent with the hypothesis that case sensitivity expands vocabulary without proportional discriminative benefit for sentiment tasks.

**Requirements:** See requirements.txt for package versions.

**Step 1**
Importing the data, reading and storing the text files. Using two lists to store the positive and negative texts separately.

In [1]:
import os

# Download the Cornell Movie Review Dataset from:
# https://www.cs.cornell.edu/people/pabo/movie-review-data/
# Extract so that data/pos/ and data/neg/ exist at the project root.

def readTexts(folderPath):
  texts = []
  for filename in os.listdir(folderPath):
    filePath = os.path.join(folderPath, filename)
    with open(filePath, "r") as file:
      text = file.read()
      texts.append(text)
  return texts

posTexts = readTexts("data/pos")
negTexts = readTexts("data/neg")

print(posTexts[6])

Mounted at /content/drive
Alan Curtis has a loud, violent sounding argument with his wife, slams out of his apartment, has a night of drinking with a mysterious lady with a large hat in a bar (run by Andrew Tombes, in a nice villainous part for a change), and returns to find his wife dead and the police, led by Thomas Gomez waiting for him. His attempts to prove his alibi - that he was with that mysterious lady - fall because everyone that he can think of (Tombes, Elisha Cook) claims there was never any such person. He ends up with no alibi, although his secretary (who secretly loves him) Ellen Raines believes him. Convicted after a trial, he is awaiting his death sentence. Raines starts going out after the truth, discovering that Gomez has some doubts of his own. She also finds an ally in a friend of Curtis, Franchot Tone, who was apparently out of town the night of the crime. Will she clear Curtis in time? THE PHANTOM LADY is based on a novel by William Irish (the great noir writer C

**Step 2** Splitting the data into training, development and test splits using *sklearn*. I used the ratio 60:20:20 respectively and used randomized shuffling as the positive and negative classes have an equal amount of texts, so stratified splitting would not have a significant difference.

In [2]:
import sklearn
import numpy
from sklearn.model_selection import train_test_split

# combining both sets of reviews to create a master list
# this contains the positive reviews set concatenated with the negative reviews set
# creating a master list of labels that corresponds with the master list of review texts
allTexts = posTexts + negTexts
labels = [1] * len(posTexts) + [0] * len(negTexts)

# using NLTK train_test_split to split the master review and label lists
# randomized shuffling with random_state = 5 to ensure that the train, development, and
#   test labels match the labels and the texts up
# the ratio is 60:20:20 respectively
xTrain, xTest, yTrain, yTest = train_test_split(allTexts, labels, test_size = 0.2, random_state = 5)
xTrain, xDev, yTrain, yDev = train_test_split(xTrain, yTrain, test_size = 0.25, random_state = 5)

# checking the sizes of each set to ensure they are as expected
print("Training sizes: ", len(xTrain), len(yTrain), "Development sizes: ", len(xDev), len(yDev), "Test sizes: ", len(xTest), len(yTest))

Training sizes:  2400 2400 Development sizes:  800 800 Test sizes:  800 800


**Step 3** Implementing BERT independently of all previous sections and experimenting with the cased and uncased versions of the model. Code developed with guidance from Hugging Face https://huggingface.co/transformers/v3.2.0/custom_datasets.html

In [ ]:
!pip install torch transformers scikit-learn
!pip install transformers[torch]
!pip install accelerate -U

In [6]:
# importing all the libraries required
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertForSequenceClassification, BertTokenizer
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset, DataLoader
import torch

# xTrain, yTrain, xDev, yDev, xTest, yTest are already defined
# defining a custom dataset class for handling data
class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # preparing data item for a given index
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        # returning the total number of instances in the dataset
        return len(self.labels)

# initialising BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-cased')

# tokenizing and encoding training, development, and test sets
trainEncodings = tokenizer(xTrain, truncation = True, padding = True, return_tensors = 'pt')
devEncodings = tokenizer(xDev, truncation = True, padding = True, return_tensors = 'pt')
testEncodings = tokenizer(xTest, truncation = True, padding = True, return_tensors = 'pt')

# creating custom datasets using the encoded data
trainDataset = CustomDataset(trainEncodings, yTrain)
devDataset = CustomDataset(devEncodings, yDev)
testDataset = CustomDataset(testEncodings, yTest)

# defining training arguments for the Trainer
training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    warmup_steps = 500,
    weight_decay = 0.01,
    logging_dir = './logs',
    logging_steps = 250,
)

# initializing BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels = 2)

# creating a Trainer instance for training the model
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = trainDataset,
    eval_dataset = devDataset
)

# training the model
trainer.train()

# evaluating on the test set
predictions = trainer.predict(testDataset)
predictedLabels = predictions.predictions.argmax(-1)
trueLabels = yTest

# calculating accuracy and other metrics
accuracy = accuracy_score(trueLabels, predictedLabels)
classificationRep = classification_report(trueLabels, predictedLabels)

print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{classificationRep}")


tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-6-0085ccb3635a>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Step,Training Loss
250,0.595300
500,0.575300
750,0.534400
1000,0.464700
1250,0.367200
1500,0.181700
1750,0.147700


<ipython-input-6-0085ccb3635a>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-6-0085ccb3635a>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-6-0085ccb3635a>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Accuracy: 0.88625
Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.91      0.89       407
           1       0.90      0.87      0.88       393

    accuracy                           0.89       800
   macro avg       0.89      0.89      0.89       800
weighted avg       0.89      0.89      0.89       800

